# PkVision — Synthetic Skeleton Training

From 89 real clips → thousands of synthetic skeleton sequences → trained ST-GCN model.

**Upload:** `skeleton_training.zip` (5.6MB)
**Runtime:** GPU (Runtime > Change runtime type > T4)

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
from google.colab import files
print('Upload skeleton_training.zip (5.6MB):')
uploaded = files.upload()
!unzip -q -o skeleton_training.zip
print('Done!')

In [ ]:
import json, math, time
import numpy as np
import torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset, random_split

kp_dir = Path('data/colab_package/keypoints')
relabels = json.load(open('data/colab_package/relabels.json'))
trick_counts = {}
for fn, tid in relabels.items():
    if tid != 'not_a_flip': trick_counts[tid] = trick_counts.get(tid, 0) + 1
print(f'Real keypoints: {len(list(kp_dir.glob("*.npy")))} files')
print(f'Classes: {len(trick_counts)}')
for tid, c in sorted(trick_counts.items(), key=lambda x: -x[1]):
    print(f'  {tid:25s} {c:3d}')

In [ ]:
# Synthetic Generator
MIRROR_PAIRS = [(1,2),(3,4),(5,6),(7,8),(9,10),(11,12),(13,14),(15,16)]
class SyntheticGenerator:
    def __init__(self, tf=64, seed=42):
        self.tf=tf; self.rng=np.random.default_rng(seed); self.real={}
    def load(self, tid, seqs): self.real[tid]=seqs
    def generate(self, tid, n=1000): return [self._aug(self.real[tid][self.rng.integers(len(self.real[tid]))].copy()) for _ in range(n)]
    def _aug(self, d):
        d=self._rsz(d,self.tf)
        if self.rng.random()<.8: d=self._spd(d,self.rng.uniform(.7,1.4))
        if self.rng.random()<.7: d=self._rot(d,self.rng.uniform(-30,30))
        if self.rng.random()<.6:
            sx,sy=self.rng.uniform(.85,1.15),self.rng.uniform(.85,1.15)
            r=d.copy(); cx=d[0].mean(1,keepdims=True); cy=d[1].mean(1,keepdims=True)
            r[0]=(d[0]-cx)*sx+cx; r[1]=(d[1]-cy)*sy+cy; d=np.clip(r,0,1)
        if self.rng.random()<.7: r=d.copy(); r[0]=np.clip(d[0]+self.rng.uniform(-.15,.15),0,1); r[1]=np.clip(d[1]+self.rng.uniform(-.15,.15),0,1); d=r
        if self.rng.random()<.5: r=d.copy(); my=d[1].mean(); r[1]=np.clip(my+(d[1]-my)*self.rng.uniform(.8,1.2),0,1); d=r
        if self.rng.random()<.8: r=d.copy(); r[:2]+=self.rng.normal(0,self.rng.uniform(.005,.02),size=(2,d.shape[1],d.shape[2])).astype(d.dtype); d=np.clip(r,0,1)
        if self.rng.random()<.5: d=self._jit(d)
        if self.rng.random()<.5: d=self._mir(d)
        if self.rng.random()<.6: r=d.copy(); r[2]=np.clip(d[2]*self.rng.uniform(.7,1.,size=(1,d.shape[1],d.shape[2])),0,1); d=r
        return self._rsz(d,self.tf)
    def _spd(self,d,f):
        C,T,V=d.shape; nT=max(4,int(T/f)); idx=np.linspace(0,T-1,nT); r=np.zeros((C,nT,V),dtype=d.dtype)
        for c in range(C):
            for v in range(V): r[c,:,v]=np.interp(idx,np.arange(T),d[c,:,v])
        return r
    def _rot(self,d,deg):
        r=d.copy(); a=math.radians(deg); co,si=math.cos(a),math.sin(a)
        cx=d[0].mean(1,keepdims=True); cy=d[1].mean(1,keepdims=True)
        x,y=d[0]-cx,d[1]-cy; r[0]=x*co-y*si+cx; r[1]=x*si+y*co+cy; return np.clip(r,0,1)
    def _jit(self,d):
        C,T,V=d.shape; r=d.copy()
        if T<5: return r
        for v in range(V):
            sh=self.rng.integers(-2,3)
            if sh!=0:
                for c in range(2): s=np.roll(d[c,:,v],sh); r[c,:,v]=s
        return r
    def _mir(self,d):
        r=d.copy(); r[0]=1.-d[0]
        for l,ri in MIRROR_PAIRS: r[:,:,l],r[:,:,ri]=r[:,:,ri].copy(),r[:,:,l].copy()
        return r
    def _rsz(self,d,t):
        C,T,V=d.shape
        if T==t: return d
        idx=np.linspace(0,T-1,t); r=np.zeros((C,t,V),dtype=d.dtype)
        for c in range(C):
            for v in range(V): r[c,:,v]=np.interp(idx,np.arange(T),d[c,:,v])
        return r
print('Generator ready')

In [ ]:
# Generate synthetic data
SAMPLES_PER_TRICK = 1000  # Change this: 500, 1000, 2000, 5000

trick_kps = {}
for fn, tid in relabels.items():
    if tid == 'not_a_flip': continue
    stem = Path(fn).stem
    kp = kp_dir / f'{stem}.npy'
    if kp.exists(): trick_kps.setdefault(tid, []).append(kp)

gen = SyntheticGenerator(tf=64, seed=42)
for tid, paths in trick_kps.items():
    seqs = [np.load(p).astype(np.float32) for p in paths if np.load(p).shape[0]==3 and np.load(p).shape[2]==17]
    if seqs: gen.load(tid, seqs)

all_data, all_labels = [], []
classes = sorted(trick_kps.keys())
c2i = {c:i for i,c in enumerate(classes)}

for tid in classes:
    if tid not in gen.real: continue
    n = min(SAMPLES_PER_TRICK, max(300, len(trick_kps[tid])*100))
    print(f'  {tid:25s} {len(trick_kps[tid]):3d} real -> {n:5d} synthetic...', end=' ')
    for s in gen.generate(tid, n=n):
        all_data.append(s); all_labels.append(c2i[tid])
    # Add real too
    for p in trick_kps[tid]:
        d = np.load(p).astype(np.float32)
        if d.shape[0]==3 and d.shape[2]==17:
            all_data.append(gen._rsz(d,64)); all_labels.append(c2i[tid])
    print('OK')

X = np.stack(all_data); y = np.array(all_labels)
print(f'\nDataset: {X.shape[0]} samples, {len(classes)} classes')

In [ ]:
# ST-GCN Model
def build_adj(n=17):
    edges=[(0,1),(0,2),(1,3),(2,4),(0,5),(0,6),(5,6),(5,7),(7,9),(6,8),(8,10),(5,11),(6,12),(11,12),(11,13),(13,15),(12,14),(14,16)]
    a=np.eye(n,dtype=np.float32)
    for i,j in edges: a[i,j]=a[j,i]=1.
    d=np.sum(a,1); di=np.where(d>0,np.power(d,-.5),0.); return np.diag(di)@a@np.diag(di)

class SGC(nn.Module):
    def __init__(s,ic,oc,adj): super().__init__(); s.register_buffer('adj',adj); s.c=nn.Conv2d(ic,oc,1); s.bn=nn.BatchNorm2d(oc); s.r=nn.ReLU(True)
    def forward(s,x):
        B,C,T,V=x.shape; xf=x.permute(0,2,1,3).reshape(B*T,C,V)
        return s.r(s.bn(s.c(torch.matmul(xf,s.adj).reshape(B,T,C,V).permute(0,2,1,3))))
class TC(nn.Module):
    def __init__(s,ic,oc,k=9): super().__init__(); s.c=nn.Conv2d(ic,oc,(k,1),padding=(k//2,0)); s.bn=nn.BatchNorm2d(oc); s.r=nn.ReLU(True)
    def forward(s,x): return s.r(s.bn(s.c(x)))
class Block(nn.Module):
    def __init__(s,ic,oc,adj):
        super().__init__(); s.s=SGC(ic,oc,adj); s.t=TC(oc,oc)
        s.res=nn.Sequential(nn.Conv2d(ic,oc,1),nn.BatchNorm2d(oc)) if ic!=oc else nn.Identity(); s.r=nn.ReLU(True)
    def forward(s,x): return s.r(s.t(s.s(x))+s.res(x))

class STGCN(nn.Module):
    def __init__(s, nc, ic=3, nj=17, h=[64,64,128,128,256]):
        super().__init__()
        adj=torch.FloatTensor(build_adj(nj)); s.ibn=nn.BatchNorm1d(ic*nj)
        ls=[]; p=ic
        for hi in h: ls.append(Block(p,hi,adj)); p=hi
        s.blocks=nn.ModuleList(ls); s.gap=nn.AdaptiveAvgPool2d(1); s.drop=nn.Dropout(.3); s.fc=nn.Linear(h[-1],nc)
    def forward(s,x):
        B,C,T,V=x.shape; x=s.ibn(x.permute(0,2,1,3).reshape(B*T,C*V)).reshape(B,T,C,V).permute(0,2,1,3)
        for b in s.blocks: x=b(x)
        return s.fc(s.drop(s.gap(x).squeeze(-1).squeeze(-1)))

model = STGCN(nc=len(classes)).to(device)
print(f'ST-GCN: {sum(p.numel() for p in model.parameters()):,} params, {len(classes)} classes')

In [ ]:
# Train
EPOCHS, BS, LR = 100, 64, 1e-3
ds = TensorDataset(torch.FloatTensor(X), torch.LongTensor(y))
vs = max(1, int(len(ds)*.15))
tds, vds = random_split(ds, [len(ds)-vs, vs], generator=torch.Generator().manual_seed(42))
tl = DataLoader(tds, batch_size=BS, shuffle=True)
vl = DataLoader(vds, batch_size=BS)
print(f'Train: {len(tds)}, Val: {len(vds)}')

crit = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
best, hist = 0., {'tl':[],'vl':[],'ta':[],'va':[]}

for ep in range(EPOCHS):
    t0=time.time(); model.train(); tls,tc,tt=0.,0,0
    for bx,by in tl:
        bx,by=bx.to(device),by.to(device); opt.zero_grad()
        o=model(bx); l=crit(o,by); l.backward(); opt.step()
        tls+=l.item()*bx.size(0); tc+=(o.argmax(1)==by).sum().item(); tt+=bx.size(0)
    sch.step(); model.eval(); vls,vc,vt=0.,0,0
    with torch.no_grad():
        for bx,by in vl:
            bx,by=bx.to(device),by.to(device); o=model(bx); l=crit(o,by)
            vls+=l.item()*bx.size(0); vc+=(o.argmax(1)==by).sum().item(); vt+=bx.size(0)
    ta,va=tc/max(tt,1),vc/max(vt,1)
    hist['tl'].append(tls/max(tt,1)); hist['vl'].append(vls/max(vt,1)); hist['ta'].append(ta); hist['va'].append(va)
    if (ep+1)%10==0 or ep==0: print(f'Epoch {ep+1:3d}/{EPOCHS} | train {tls/max(tt,1):.4f} {ta:.0%} | val {vls/max(vt,1):.4f} {va:.0%} | {time.time()-t0:.1f}s')
    if va>=best: best=va; torch.save({'model_state_dict':model.state_dict(),'classes':classes,'epoch':ep,'val_acc':va,'config':{'num_classes':len(classes),'in_channels':3,'num_joints':17}},'stgcn_synthetic.pt')
print(f'\nBest: {best:.0%}')

In [ ]:
# Training curves
import matplotlib.pyplot as plt
fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5))
a1.plot(hist['tl'],label='Train');a1.plot(hist['vl'],label='Val');a1.set_title('Loss');a1.legend();a1.grid(alpha=.3)
a2.plot([a*100 for a in hist['ta']],label='Train');a2.plot([a*100 for a in hist['va']],label='Val');a2.set_title('Accuracy (%)');a2.legend();a2.grid(alpha=.3)
plt.tight_layout();plt.show()

In [ ]:
# Per-class accuracy
model.eval(); ap,at=[],[]
with torch.no_grad():
    for bx,by in vl: ap.extend(model(bx.to(device)).argmax(1).cpu().tolist()); at.extend(by.tolist())
print('Per-class accuracy:'); print('-'*45)
for i,c in enumerate(classes):
    m=[j for j,t in enumerate(at) if t==i]
    if m: acc=sum(1 for j in m if ap[j]==i)/len(m); print(f'  {c:25s} {acc:5.0%} {"#"*int(acc*20)} ({sum(1 for j in m if ap[j]==i)}/{len(m)})')

In [ ]:
# Download
files.download('stgcn_synthetic.pt')
print('Save as: pkvision/data/models/stgcn_synthetic.pt')